# 02 — Running MD Simulations
Equilibrate and run production MD on a prepared system.
Uses GPU acceleration (CUDA) for improved performance.

## Workflow
1. NVT equilibration (heating to target temperature)
2. NPT equilibration (pressure coupling)
3. Production MD
4. (Optional) Steered MD or umbrella sampling

### Configuration — Edit these values

In [ ]:
import os
import subprocess
from pathlib import Path

# ── System paths ─────────────────────────────────────────────
PREP_DIR = "data/my_protein_prep"       # From preparation step
TOPOLOGY = f"{PREP_DIR}/topol.top"
INITIAL_STRUCTURE = f"{PREP_DIR}/em.gro" # Post-EM structure
OUTPUT_DIR = f"data/my_protein_md"

# ── Simulation parameters ────────────────────────────────────
TEMPERATURE = 310.0                     # Kelvin (310 for mammalian)
PRESSURE = 1.0                           # bar
PRODUCTION_TIME_NS = 100                 # Production run length
DT_PS = 0.002                            # Timestep (ps) — 2 fs standard

# ── GPU settings ─────────────────────────────────────────────
GPU_ID = 0                               # GPU device index
NSTLIST = 20                             # Neighbor list update frequency

# ── Membrane simulation? ─────────────────────────────────────
MEMBRANE_SIMULATION = False              # Match preparation
SEMIISOTROPIC = MEMBRANE_SIMULATION      # Semi-isotropic for membranes

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/mdp", exist_ok=True)

def run_gmx(cmd, desc=""):
    print(f"  [{desc}]" if desc else f"  $ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  WARNING: rc={result.returncode}")
        print(result.stderr[-300:] if result.stderr else "")
    return result

## Step 1: NVT Equilibration (heating)
Heat the system to target temperature under constant volume.
Uses velocity-rescale thermostat with separate protein/non-protein coupling.

In [ ]:
nvt_mdp = f"""; NVT equilibration
integrator       = md
dt              = {DT_PS}
nsteps          = 250000  ; 500 ps
nstxout         = 5000
nstvout         = 5000
nstenergy       = 500
nstlog          = 500
nstxout-compressed = 5000
compressed-x-grps = System

cutoff-scheme   = Verlet
ns_type         = grid
nstlist         = {NSTLIST}
rlist           = 1.2
coulombtype     = PME
rcoulomb        = 1.2
vdwtype         = cut-off
rvdw            = 1.2
DispCorr        = EnerPres

tcoupl          = V-rescale
tc-grps         = Protein Non-Protein
tau_t           = 0.5 0.5
ref_t           = {TEMPERATURE} {TEMPERATURE}
pcoupl          = no
pbc             = xyz

constraints             = h-bonds
constraint-algorithm    = lincs
lincs-order             = 4
lincs-iter              = 1

gen_vel         = yes
gen_temp        = {TEMPERATURE}
gen_seed        = -1
"""
with open(f"{OUTPUT_DIR}/mdp/nvt.mdp", 'w') as f:
    f.write(nvt_mdp)
print("NVT MDP written.")

In [ ]:
run_gmx(f"gmx grompp -f {OUTPUT_DIR}/mdp/nvt.mdp -c {INITIAL_STRUCTURE} -r {INITIAL_STRUCTURE} "
        f"-p {TOPOLOGY} -o {OUTPUT_DIR}/nvt.tpr -po {OUTPUT_DIR}/mdp/nvt_out.mdp -maxwarn 2",
        desc="grompp: NVT")

print("\nRunning NVT equilibration on GPU... (will take ~2-10 min depending on system size)")
result = run_gmx(f"gmx mdrun -v -deffnm {OUTPUT_DIR}/nvt -s {OUTPUT_DIR}/nvt.tpr "
                f"-gpu_id {GPU_ID} -nb gpu -pme gpu -bonded cpu -update gpu",
                desc="mdrun: NVT")

In [ ]:
# Quick check: temperature stability
import matplotlib.pyplot as plt
import numpy as np

run_gmx(f"echo -e '16\n0\n' | gmx energy -f {OUTPUT_DIR}/nvt.edr -o {OUTPUT_DIR}/nvt_temp.xvg",
        desc="extract temperature")

try:
    import pandas as pd
    data = pd.read_csv(f"{OUTPUT_DIR}/nvt_temp.xvg", comment='#', comment='@',
                       delim_whitespace=True, header=None, names=['time_ps','temp'])
    plt.figure(figsize=(10, 4))
    plt.plot(data['time_ps'], data['temp'], lw=0.5)
    plt.axhline(TEMPERATURE, color='red', ls='--', alpha=0.5, label=f'Target {TEMPERATURE}K')
    plt.xlabel('Time (ps)')
    plt.ylabel('Temperature (K)')
    plt.title('NVT Equilibration — Temperature')
    plt.legend()
    plt.grid(alpha=0.3)
    mean_temp = data['temp'].iloc[-100:].mean()
    print(f"\nMean temperature (last 100 steps): {mean_temp:.1f} K")
except Exception as e:
    print(f"Could not plot: {e}")

## Step 2: NPT Equilibration
Equilibrate pressure under constant temperature.
Use semiisotropic pressure coupling for membrane systems.

In [ ]:
pcouple_type = "semiisotropic" if SEMIISOTROPIC else "berendsen"
pcouple_geom = "xyz" if not SEMIISOTROPIC else "semi-isotropic"

npt_mdp = f"""; NPT equilibration
integrator       = md
dt              = {DT_PS}
nsteps          = 500000  ; 1 ns
nstxout         = 5000
nstenergy       = 500
nstlog          = 500
nstxout-compressed = 5000
compressed-x-grps = System

cutoff-scheme   = Verlet
ns_type         = grid
nstlist         = {NSTLIST}
rlist           = 1.2
coulombtype     = PME
rcoulomb        = 1.2
vdwtype         = cut-off
rvdw            = 1.2
DispCorr        = EnerPres

tcoupl          = V-rescale
tc-grps         = Protein Non-Protein
tau_t           = 0.5 0.5
ref_t           = {TEMPERATURE} {TEMPERATURE}

pcoupl          = {pcouple_type}
pcoupltype      = isotropic
tau_p           = 5.0
ref_p           = {PRESSURE}
compressibility = 4.5e-5
refcoord_scaling = com

pbc             = xyz
constraints             = h-bonds
constraint-algorithm    = lincs
lincs-order             = 4

gen_vel         = no
"""

with open(f"{OUTPUT_DIR}/mdp/npt.mdp", 'w') as f:
    f.write(npt_mdp)
print("NPT MDP written.")

In [ ]:
run_gmx(f"gmx grompp -f {OUTPUT_DIR}/mdp/npt.mdp -c {OUTPUT_DIR}/nvt.gro -r {OUTPUT_DIR}/nvt.gro "
        f"-t {OUTPUT_DIR}/nvt.cpt -p {TOPOLOGY} -o {OUTPUT_DIR}/npt.tpr -maxwarn 2",
        desc="grompp: NPT")

print("\nRunning NPT equilibration on GPU...")
run_gmx(f"gmx mdrun -v -deffnm {OUTPUT_DIR}/npt -s {OUTPUT_DIR}/npt.tpr "
        f"-gpu_id {GPU_ID} -nb gpu -pme gpu -bonded cpu -update gpu",
        desc="mdrun: NPT")

In [ ]:
# Check density convergence
run_gmx(f"echo -e '24\n0\n' | gmx energy -f {OUTPUT_DIR}/npt.edr -o {OUTPUT_DIR}/npt_density.xvg",
        desc="extract density")

import pandas as pd
data = pd.read_csv(f"{OUTPUT_DIR}/npt_density.xvg", comment='#', comment='@',
                   delim_whitespace=True, header=None, names=['time_ps','density'])
mean_rho = data['density'].iloc[-100:].mean()
print(f"Mean density (last 100 steps): {mean_rho:.1f} kg/m^3")
print(f"Expected (TIP3P+protein): ~1000-1050 kg/m^3")

plt.figure(figsize=(10, 4))
plt.plot(data['time_ps'], data['density'], lw=0.5)
plt.xlabel('Time (ps)')
plt.ylabel('Density (kg/m^3)')
plt.title('NPT Equilibration — Density')
plt.grid(alpha=0.3)
plt.show()

## Step 3: Production MD
Run the production simulation. For long runs, consider checkpoint/restart.
Run in background and monitor progress with `03_monitoring.ipynb`.

In [ ]:
nsteps_prod = int(PRODUCTION_TIME_NS * 1000 / DT_PS)
print(f"Production: {PRODUCTION_TIME_NS} ns = {nsteps_prod} steps")
print(f"Estimated time on TITAN Xp:")
print(f"  ~10-20 ns/day for a 50k-atom soluble system")
print(f"  ~5-10 ns/day for a 100k-atom membrane system")

prod_mdp = f"""; Production MD
integrator       = md
dt              = {DT_PS}
nsteps          = {nsteps_prod}
nstxout         = 0
nstvout         = 0
nstenergy       = 5000
nstlog          = 5000
nstxout-compressed = 50000  ; Save every 100 ps
compressed-x-grps = System

cutoff-scheme   = Verlet
ns_type         = grid
nstlist         = {NSTLIST}
rlist           = 1.2
coulombtype     = PME
rcoulomb        = 1.2
vdwtype         = cut-off
rvdw            = 1.2
DispCorr        = EnerPres

tcoupl          = V-rescale
tc-grps         = Protein Non-Protein
tau_t           = 0.5 0.5
ref_t           = {TEMPERATURE} {TEMPERATURE}

pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 5.0
ref_p           = {PRESSURE}
compressibility = 4.5e-5
refcoord_scaling = com

pbc             = xyz
constraints             = h-bonds
constraint-algorithm    = lincs
lincs-order             = 4

gen_vel         = no
"""

with open(f"{OUTPUT_DIR}/mdp/prod.mdp", 'w') as f:
    f.write(prod_mdp)
print("Production MDP written.")

In [ ]:
run_gmx(f"gmx grompp -f {OUTPUT_DIR}/mdp/prod.mdp -c {OUTPUT_DIR}/npt.gro -t {OUTPUT_DIR}/npt.cpt "
        f"-p {TOPOLOGY} -o {OUTPUT_DIR}/prod.tpr -maxwarn 2",
        desc="grompp: production")

print("\n" + "="*60)
print("Ready to run production MD!")
print("="*60)
print(f"\nRun this command in a terminal or submit to a scheduler:")
print(f"\n  conda activate gromacs_env")
print(f"  cd {OUTPUT_DIR}")
print(f"  gmx mdrun -v -deffnm prod -s prod.tpr \\")
print(f"       -gpu_id {GPU_ID} -nb gpu -pme gpu -bonded cpu -update gpu")
print(f"\nOr run in this notebook with background process:")
print(f"  import subprocess")
print(f"  proc = subprocess.Popen([...])  # see below")

In [ ]:
# Uncomment to launch production run directly from here
# This will block the notebook — use 03_monitoring.ipynb instead

# import subprocess, sys
# cmd = f"gmx mdrun -v -deffnm {OUTPUT_DIR}/prod -s {OUTPUT_DIR}/prod.tpr "\
#       f"-gpu_id {GPU_ID} -nb gpu -pme gpu -bonded cpu -update gpu -cpi {OUTPUT_DIR}/prod.cpt"
# proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
#                          bufsize=1, universal_newlines=True)
# for line in proc.stdout:
#     sys.stdout.write(line)
# proc.wait()
# print(f"Production completed (exit code {proc.returncode})")

## Next Steps
- Monitor the simulation with `03_monitoring.ipynb`
- Analyze results with `04_analysis.ipynb`